# Connect to Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


# Install packages

In [2]:
!pip install ultralytics albumentations -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 40.0 MB/s eta 0:00:00


# Unzip dataset

In [3]:
ROOT_DIR = '/content/gdrive/My Drive/Colab Notebooks/Object detection'
ZIP_NAME = 'dataset.zip'

import os
zip_path = f'{ROOT_DIR}/{ZIP_NAME}'
assert os.path.exists(zip_path), f'File not found: {zip_path}'

print(f'Loading {ZIP_NAME}...')
!unzip -q "{zip_path}" -d /content/
print('Loading finished.')

Loading dataset.zip...
Loading finished.


# Split dataset
Use only 1 time for splitting dataset to file train/val/test

In [ ]:
# import random
# import shutil
# from pathlib import Path

# random.seed(42)
# DATA = Path('/content/data')

# img_train = DATA / 'images' / 'train'
# lbl_train = DATA / 'labels' / 'train'
# img_val   = DATA / 'images' / 'val'
# lbl_val   = DATA / 'labels' / 'val'
# img_test  = DATA / 'images' / 'test'
# lbl_test  = DATA / 'labels' / 'test'

# # Create val and test folder(if not exist)
# for d in [img_val, lbl_val, img_test, lbl_test]:
#   d.mkdir(parents=True, exist_ok=True)

# all_imgs = sorted([
#     f for f in img_train.glob('*.*')
#     if f.suffix.lower() == '.jpg'
#     and '_aug' not in f.stem
# ])

# random.shuffle(all_imgs)
# n_total = len(all_imgs)
# n_val = int(n_total*0.2)
# n_test = int(n_total*0.1)

# val_imgs  = all_imgs[:n_val]
# test_imgs = all_imgs[n_val:n_val + n_test]

# print(f'Total                 : {n_total}')
# print(f'Train                 : {n_total - n_val - n_test}')
# print(f'Validation            : {n_val}')
# print(f'Test                  : {n_test}')
# print('Moving files...')

# def move_pair(img_path, dst_img_dir, dst_lbl_dir):
#   lbl_path = lbl_train/(img_path.stem + '.txt')
#   shutil.move(str(img_path), str(dst_img_dir / img_path.name))
#   if lbl_path.exists():
#     shutil.move(str(lbl_path), str(dst_lbl_dir / lbl_path.name))

# for img in val_imgs:
#   move_pair(img, img_val, lbl_val)

# for img in test_imgs:
#   move_pair(img, img_test, lbl_test)

# # Checking for sure
# print('\n-------After splitting------')
# for split in ['train', 'val', 'test']:
#     n_img = len(list((DATA / 'images' / split).glob('*.*')))
#     n_lbl = len(list((DATA / 'labels' / split).glob('*.txt')))
#     print(f'  {split:6s}: {n_img} image | {n_lbl} label')


Total                 : 1000
Train                 : 700
Validation            : 200
Test                  : 100
Moving files...

-------After splitting------
  train : 2044 image | 1990 label
  val   : 340 image | 316 label
  test  : 170 image | 157 label


# Zip dataset
Only use 1 time after splitting dataset.

In [ ]:
# import shutil

# print('Zipping dataset...')

# shutil.make_archive(
#     base_name='/content/dataset',
#     format='zip',
#     root_dir='/content',
#     base_dir='data'
# )

# print('Copying to Google Drive...')

# shutil.copy(
#     '/content/dataset.zip',
#     '/content/gdrive/My Drive/Colab Notebooks/Object detection/dataset.zip'
# )

# print('DONE! Old file has been changed.')


Zipping dataset...
Copying to Google Drive...
DONE! Old file has been changed.


# Path config

In [4]:
from pathlib import Path

DATA = Path('/content/data')
CONFIG = Path('/content/gdrive/My Drive/Colab Notebooks/Object detection/google_colab.yaml')
OUTPUT = Path('/content/gdrive/MyDrive/Colab Notebooks/Object detection/runs')
OUTPUT.mkdir(parents=True, exist_ok=True)

assert CONFIG.exists(), 'google_colab.yaml not found.'


# Augmentation

In [5]:
import cv2
import albumentations as A

augment_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.GaussNoise(std_range=(0.01, 0.05), p=0.4),
    A.GaussianBlur(blur_limit=3, p=0.3),
    A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=20, val_shift_limit=20, p=0.3)
], bbox_params = A.BboxParams(
    format='yolo',
    label_fields=['class_labels'],
    min_visibility=0.3,
    clip=True
))

def read_label(path):
  boxes, labels = [], []
  if not Path(path).exists():
    return boxes, labels
  with open(path) as f:
    for line in f:
      parts = list(map(float, line.strip().split()))
      if len(parts) == 5:
        cls, x, y, w, h = parts
        x = max(0.0, min(1.0, x))
        y = max(0.0, min(1.0, y))
        w  = max(0.0, min(1.0, w))
        h  = max(0.0, min(1.0, h))
        boxes.append([x, y, w, h])
        labels.append(int(cls))
  return boxes, labels


def save_label(path, boxes, labels):
  with open(path, 'w') as f:
    for box, cls in zip(boxes, labels):
      f.write(f'{cls} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}\n')

def augment_dataset(img_dir, lbl_dir, n_augment=3):
  img_dir = Path(img_dir)
  lbl_dir = Path(lbl_dir)

  # Remove the old augmented file
  old_files = list(img_dir.glob('*_aug*')) + list(lbl_dir.glob('*_aug*'))
  for f in old_files:
    f.unlink()
  print(f'Removed {len(old_files)} old augmented files.')
  img_files = list(img_dir.glob('*.jpg'))

  print(f'Augmenting {n_augment} times')

  created = 0
  for img_path in img_files:
    img = cv2.imread(str(img_path))
    if img is None:
      continue
    boxes, labels = read_label(str(lbl_dir / (img_path.stem + '.txt')))
    if not boxes:
      continue
    for i in range(n_augment):
      try:
        result = augment_pipeline(image=img, bboxes=boxes, class_labels=labels)
        if not result['bboxes']:
          continue
        save_name = f'{img_path.stem}_aug{i}'
        cv2.imwrite(str(img_dir / f'{save_name}.jpg'), result['image'])
        save_label(str(lbl_dir / f'{save_name}.txt'), result['bboxes'], result['class_labels'])
        created += 1
      except Exception as e:
        print(f'Error {img_path.name}: {e}')
  orig = len(img_files)
  print(f'DONE! Original: {orig} | Added: {created} | Total: {orig + created}')


augment_dataset(
    img_dir=DATA / 'images' / 'train',
    lbl_dir=DATA / 'labels' / 'train',
    n_augment=3
)


Removed 2688 old augmented files.
Augmenting 3 times
DONE! Original: 700 | Added: 1929 | Total: 2629


# Train YOLOv8

In [6]:
import os
from ultralytics import YOLO

# Load a model
model = YOLO("yolov8n.pt")

# Use the model
results = model.train(
    data=str(CONFIG),
    epochs=100,
    imgsz=1280,
    batch=8,
    patience=20,
    save=True,
    project=str(OUTPUT),
    name='run1',
    exist_ok=True,
    augment=True,
    plots=True,
    verbose=True
)


BEST_MODEL_PATH = Path(results.save_dir) / 'weights' / 'best.pt'
print(f'\nTraining is done!')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.49 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/gdrive/My Drive/Colab Notebooks/Object detection/google_colab.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half

# Train (next)
Only use in case your training is interrupted or you want to train more epoch in this model.

In [ ]:
from ultralytics import YOLO

model = YOLO(str(OUTPUT / 'run1/weights/last.pt'))
results = model.train(resume=True)

BEST_MODEL_PATH = Path(results.save_dir) / 'weights' / 'best.pt'

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/gdrive/My Drive/Colab Notebooks/Object detection/google_colab.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_

# Evaluating in Validation

In [7]:
best_model = YOLO(str(BEST_MODEL_PATH))

v_metrics = best_model.val(data=str(CONFIG), split='val')

print('===== RESULT IN VALIDATION =====')
print(f'mAP@0.5      : {v_metrics.box.map50:.4f}  ({v_metrics.box.map50*100:.1f}%)')
print(f'mAP@0.5:0.95 : {v_metrics.box.map:.4f}')
print(f'Precision    : {v_metrics.box.p.mean():.4f}')
print(f'Recall       : {v_metrics.box.r.mean():.4f}')



Ultralytics 8.4.49 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1807.5±316.2 MB/s, size: 64.8 KB)
val: Scanning /content/data/labels/val.cache... 316 images, 25 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 340/340 101.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 1.6it/s 14.0s
                   all        340       2963      0.759      0.744      0.781       0.45
               bo thon         28         32      0.784      0.875       0.88      0.653
                bo map        260       2028      0.806      0.792      0.835      0.381
                 bo to        238        903      0.687      0.566      0.627      0.316
Speed: 9.1ms preprocess, 14.6ms inference, 0.0ms loss, 2.5ms postprocess per image
Results saved to /content/runs/detect/val
===== RESULT IN

# Evaluating in Test

In [9]:
t_metrics = best_model.val(data=str(CONFIG), split='test')

print('===== RESULT IN TEST =====')
print(f'mAP@0.5  : {t_metrics.box.map50:.4f}  ({t_metrics.box.map50*100:.1f}%)')
print(f'Precision: {t_metrics.box.p.mean():.4f}')
print(f'Recall   : {t_metrics.box.r.mean():.4f}')

CLASS_NAMES = ['bo thon', 'bo map', 'bo to']
for i, c in enumerate(CLASS_NAMES):
  try:
    ap = t_metrics.box.ap[i]
    print(f'{c:10s}: AP@0.5:0.95 : {ap:.4f} ({ap*100:.1f}%)')
  except:
    pass


Ultralytics 8.4.49 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1407.6±428.5 MB/s, size: 95.8 KB)
val: Scanning /content/data/labels/test.cache... 157 images, 14 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 170/170 31.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.3it/s 8.3s
                   all        170       1700      0.795      0.777      0.829      0.409
               bo thon         14         18      0.826      0.944      0.953      0.479
                bo map        127        926      0.829      0.805      0.875      0.446
                 bo to        104        756      0.729      0.581       0.66      0.302
Speed: 12.4ms preprocess, 12.7ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to /content/runs/detect/val-3
===== RESULT IN TEST =====
mAP@0.5  : 0.8292  (82.9%)
Precision: 0.7945
Recall   : 0.7767
bo 

# Predict
This part is implemented in local.